# SIA P1 — Device-Actions LoRA Training (Google Colab, free T4)

Fine-tunes the SIA device-actions LoRA adapter on a **free Colab T4 GPU** —
no Google Cloud project, no GPU quota, no billing.

**Before running:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU**, then Save.

## Step 1 — Confirm the T4 GPU is attached

In [ ]:
!nvidia-smi

## Step 2 — Get the code + data
`train_gcp.py` and the dataset are on `main`.

In [ ]:
import os
os.chdir('/content')
!rm -rf SIA
!git clone https://github.com/skmandal3240/SIA.git
os.chdir('SIA')
print('cwd:', os.getcwd())

## Step 3 — Install training deps

In [ ]:
!pip install -q -r sia-lab/posttrain/requirements-train.txt

## Step 4 — Validate data + config (quick, no training yet)

In [ ]:
!python3 sia-lab/posttrain/train_gcp.py --dry-run

## Step 5 — Train on the T4
Public `unsloth/Llama-3.2-1B-Instruct` base (no Hugging Face login needed).
Output stays local in Colab; `--merge` also writes a merged fp16 model.

In [ ]:
!python3 sia-lab/posttrain/train_gcp.py \
    --base unsloth/Llama-3.2-1B-Instruct \
    --train sia-lab/posttrain/data/device_actions_train.json \
    --val   sia-lab/posttrain/data/device_actions_val.json \
    --output-dir sia-lab/posttrain/outputs/device_actions_lora \
    --epochs 3 --merge

## Step 6 — Held-out tool-call accuracy

In [ ]:
import json
print(json.load(open('sia-lab/posttrain/outputs/device_actions_lora/eval.json')))

## Step 7 — Download the trained adapter
Zips the adapter and downloads it to your machine.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('sia_device_actions_lora', 'zip', 'sia-lab/posttrain/outputs/device_actions_lora')
files.download('sia_device_actions_lora.zip')